In [1]:
import pandas as pd
import openpyxl
from IPython.display import display
import xlsxwriter
import numpy as np
from IPython.display import display, Image, HTML

### 所需要的所有文件：
# 统计周期内产品核算价汇总
# 所有11开头的物料导出MARA
# 两个月出一次，下一次是11月


### 对照关系，每次都需要更新

In [87]:
#指定月份
month = '202511'
month_name = '11月'
year_name = '2025年'
income_year_name = '2024年发货收入'
#指定时间范围
start_date = pd.Timestamp('2025-01-01')
end_date = pd.Timestamp('2025-10-31')
#每年的物料基准数量--年度更新
productgroup_starting_map = {'油烟机':11823
, '灶具':6485
, '蒸烤微合计':6655
, '灶集成':2060
, '烹饪合计':15200
, '水槽洗碗机':4088
, '嵌入式洗碗机':2936
, '洗碗机合计':7024
, '家用净水机':2083
, '热水器':4352
, '净热合计':6435
, '消毒柜':2589
, '国内产品线合计':43071
, '海外合计':7547
}

#发货收入基准，年度更新
productgroup_sales_map = {'油烟机':760275
, '灶具':364114
, '蒸烤微合计':117582
, '灶集成':51586
, '烹饪合计':533281
, '水槽洗碗机':70840
, '嵌入式洗碗机':129987
, '洗碗机合计':200827
, '家用净水机':21298
, '热水器':50544
, '净热合计':71842
, '消毒柜':51444
, '国内产品线合计':1617669
}

target_map ={
    '油烟机': 58.5,
    '灶具': 49.52,
    '蒸烤微合计':16.29,
    '灶集成': 24.89,
    '烹饪合计':31.49,
    '水槽洗碗机': '/',
    '嵌入式洗碗机': '/',
    '洗碗机合计': 25.73,
    '家用净水机': 9.2,
    '热水器': 10.45,
    '净热合计': 10.04,
    '消毒柜': 14.5,
    '国内产品线合计': 33.8
}

beizhu = {
    '油烟机': '不含ODM、空调/IMES',
    '灶具': '不含火炬、智能灶',
    '蒸烤微合计':'含烹饪机',
    '灶集成': '不含灶消',
    '烹饪合计':'不含灶消',
    '水槽洗碗机': '/',
    '嵌入式洗碗机': '/',
    '洗碗机合计': '/',
    '家用净水机': '/',
    '热水器': '含两用炉',
    '净热合计': '/',
    '消毒柜': '/',
    '国内产品线合计': '/',
    '海外合计':'/'
}
#依据规则构造出所有的对照关系，这里是用于物料的对照

productgroup_map={
    '油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪合计':['灶具','烤箱', '蒸箱', '微波炉', '蒸烤烹饪机', '蒸烤微烹饪机','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '水槽洗碗机': ['水槽洗碗机'],
    '嵌入式洗碗机': ['嵌入式洗碗机'],
    '洗碗机合计': ['水槽洗碗机','嵌入式洗碗机'],
    '家用净水机': ['家用净水机'],
    '热水器': ['热水器','两用炉'],
    '净热合计': ['家用净水机','热水器','两用炉'],
    '消毒柜': ['消毒柜'],
    '国内产品线合计': ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','水槽洗碗机','嵌入式洗碗机','家用净水机','热水器','两用炉','消毒柜']
}

product_dict = {
'B1':'橱柜',                
'B2':'集成家居',            
'F1':'服务产品',            
'Y1':'气电烹饪灶（智能灶）',
'Y2':'下排集成灶',          
'Y3':'集成式洗碗机',        
'Y4':'灶烤烹饪机',          
'Y5':'集成油烟机',          
'Y6':'蒸烤微集成烹饪中心',  
'Y7':'灶蒸烤微烹饪机',      
'Z1':'吸油烟机',            
'Z2':'灶具',                
'Z3':'消毒柜',              
'Z4':'微波炉',              
'Z5':'蒸箱',                
'Z6':'烤箱',                
'Z7':'热水器',              
'Z8':'水槽洗碗机',          
'Z9':'保温箱',              
'ZA':'蒸锅',                
'ZC':'橱柜',                
'ZD':'家用净水机',          
'ZE':'大厨管家',            
'ZF':'服务',                
'ZG':'空壳',                
'ZH':'料理机',              
'ZJ':'新风空净',            
'ZK':'蒸微',                
'ZL':'蒸烤烹饪机',          
'ZM':'灶蒸烤烹饪机',        
'ZN':'灶消烹饪机',          
'ZP':'灶蒸烹饪机',          
'ZQ':'蒸烤微烹饪机',        
'ZR':'商用净水机',          
'ZS':'嵌入式洗碗机',        
'ZT':'两用炉',              
'ZU':'外置循环系统',        
'ZV':'家用冰箱',            
'ZW':'家用厨余处理机',      
'ZX':'地面手持式清洁机',    
'ZY':'地面机器人清洁机',    
'ZZ':'其他',
}


### 2025年6月收入（每次更新收入表路径，和sheet名）,由单型号中输出

In [35]:
# 物流和财务的发货收入中是包含了1012开头的样机的，PLM中是只有成品的，所以这里样机的产品组是空的，样机不在我们报告的统计范围内
sale_income = pd.read_excel(rf'D:\000物料报表\{month}\单物料产值\统计周期内产品核算价汇总.xlsx')
核算_col = '核算价'
sale_income[f'{核算_col}'] = sale_income[f'{核算_col}'].fillna(0)
sale_income = sale_income.replace('NaN',0)
sale_income_grouped = sale_income.groupby('产品组').agg({f'{核算_col}':sum}).rename(columns={f'{核算_col}':'总收入'}).reset_index()
sale_income_grouped_map = dict(zip(sale_income_grouped['产品组'],sale_income_grouped['总收入']))

productgroup2_income_map = {}
for k,v in productgroup_map.items():
    productgroup2_income_map[k] = 0
    for item in v:
        productgroup2_income_map[k] += sale_income_grouped_map.get(item,0)/10000
productgroup2_income_map


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\2002267409.py:6: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  sale_income_grouped = sale_income.groupby('产品组').agg({f'{核算_col}':sum}).rename(columns={f'{核算_col}':'总收入'}).reset_index()


{'油烟机': 737320.8298,
 '灶具': 366186.4071,
 '蒸烤微合计': 114481.049,
 '灶集成': 64578.592,
 '烹饪合计': 545246.0481,
 '水槽洗碗机': 76989.7925,
 '嵌入式洗碗机': 142664.318,
 '洗碗机合计': 219654.1105,
 '家用净水机': 21954.001,
 '热水器': 48246.6023,
 '净热合计': 70200.6033,
 '消毒柜': 41231.075,
 '国内产品线合计': 1613652.6666999992}

In [110]:
#先把去年的数据处理了
df_2024 = pd.read_excel(fr"D:\000物料报表\202501\单物料产值\物料精简.XLSX")
len(df_2024)

60200

In [111]:
def clean(df):
    list_drop = []
    df = df.astype(str)
    df['删除原因'] = '空'
    for i in range(len(df)):
        if 'ZD70' in df.loc[i,'物料描述']:
            list_drop.append(i)
            df.loc[i,'删除原因'] = 'ODM'
        if df.loc[i,'图号'][:4] == 'LAFS':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '火炬'
        if df.loc[i,'图号'][:4] == 'LAFT':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '火炬'
        if df.loc[i,'物料编码'][:8] == '11010022':
            list_drop.append(i)
            df.loc[i,'删除原因'] = 'IMES'  
        if df.loc[i,'物料编码'][:8] == '11010027':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '空调油烟机'  
        if df.loc[i,'图号'][:4] == 'PACA':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '净存机'
        if df.loc[i,'产品组'] == 'ZN':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '灶消'
    df['产品组描述'] = df['产品组'].map(product_dict)
    df['国内/海外'] = df['国内/海外'].apply(lambda x:'国内' if x=='20' else ('海外' if x=='10' else '空'))
    df = df.fillna('空')
    df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']] = df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']].replace('nan','空')
    df_not_del = df[(df["删除原因"] == "空")&(df["是否虚拟物料"] == "空")]
    return df_not_del

In [112]:
df_2024 = clean(df_2024)
len(df_2024)

58832

In [113]:
df_2024_live = df_2024[(df_2024["是否冻结"] == "空")&(df_2024['创建日期']<='2024-12-31')].reset_index(drop=True)
len(df_2024_live)

58586

In [114]:
df_2024_live['国内/海外'].value_counts()

国内/海外
国内    51039
海外     7547
Name: count, dtype: int64

In [133]:
df_2024_result = pd.DataFrame()
df_2024_result['产品集合'] = productgroup_map.keys()
for index,row in df_2024_result.iterrows():
    df_2024_result.loc[index,'2024年基准'] = df_2024_live[(df_2024_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2024_live['国内/海外']=='国内')]['物料编码'].nunique()
    df_2024_result.loc[index,'2024年基准_自制'] = df_2024_live[(df_2024_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2024_live['国内/海外']=='国内')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='10.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2024年基准_外购'] = df_2024_live[(df_2024_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2024_live['国内/海外']=='国内')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='20.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2024年基准_外协'] = df_2024_live[(df_2024_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2024_live['国内/海外']=='国内')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='30.0')]['物料编码'].nunique()
df_2024_result

,产品集合,2024年基准,2024年基准_自制,2024年基准_外购,2024年基准_外协
0,油烟机,11823.0,4188.0,7184.0,448.0
1,灶具,6485.0,1224.0,5196.0,65.0
2,蒸烤微合计,6655.0,1445.0,5006.0,203.0
3,灶集成,2060.0,603.0,1375.0,82.0
4,烹饪合计,15200.0,3272.0,11577.0,350.0
5,水槽洗碗机,4088.0,520.0,3464.0,96.0
6,嵌入式洗碗机,2936.0,376.0,2398.0,162.0
7,洗碗机合计,7024.0,896.0,5862.0,258.0
8,家用净水机,2083.0,137.0,1868.0,78.0
9,热水器,4352.0,606.0,3644.0,102.0


In [125]:
df_2025= pd.read_excel(r"D:\000物料报表\202512\单物料产值\物料精简原始数据12月.XLSX")

In [126]:
def clean(df):
    list_drop = []
    df = df.astype(str)
    df['删除原因'] = '空'
    for i in range(len(df)):
        if 'ZD70' in df.loc[i,'物料描述']:
            list_drop.append(i)
            df.loc[i,'删除原因'] = 'ODM'
        if df.loc[i,'图号'][:4] == 'LAFS':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '火炬'
        if df.loc[i,'图号'][:4] == 'LAFT':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '火炬'
        if df.loc[i,'物料编码'][:8] == '11010022':
            list_drop.append(i)
            df.loc[i,'删除原因'] = 'IMES'  
        if df.loc[i,'物料编码'][:8] == '11010027':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '空调油烟机'  
        if df.loc[i,'图号'][:4] == 'PACA':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '净存机'
        if df.loc[i,'产品组'] == 'ZN':
            list_drop.append(i)
            df.loc[i,'删除原因'] = '灶消'
    df['产品组描述'] = df['产品组'].map(product_dict)
    df['国内/海外'] = df['国内/海外'].apply(lambda x:'国内' if x=='20.0' else ('海外' if x=='10.0' else '空'))
    df = df.fillna('空')
    df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']] = df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']].replace('nan','空')
    df_not_del = df[(df["删除原因"] == "空")&(df["是否虚拟物料"] == "空")]
    return df_not_del

In [127]:
df_2025 = clean(df_2025)

In [134]:
df_2025_live = df_2025[(df_2025['是否冻结']=='空')&(df_2025['创建日期']<='2025-11-31')]
df_2025_live

,物料编码,创建日期,是否冻结,图号,产品组,跨工厂物料状态,是否虚拟物料,制造方式,国内/海外,是否涉及认证一致性管控,整机认证地区,整机认证类型,整机海外认证标准,物料描述,删除原因,产品组描述
0,1101000100410,2018-07-10,空,1-FAA-A106,Z1,空,空,10.0,国内,nan,nan,nan,nan,线夹(CXW-128-Q2),空,吸油烟机
1,1101000100470,2018-07-10,空,1-FAA-A221,Z1,空,空,20.0,国内,nan,nan,nan,nan,网圈(CXW-128-Q2),空,吸油烟机
2,1101000100480,2018-07-10,空,1-FAA-A222,Z1,空,空,20.0,国内,nan,nan,nan,nan,网罩内圈(CXW-128-Q2),空,吸油烟机
3,1101000100510,2018-07-10,空,1-FAA-A302,Z1,空,空,20.0,国内,nan,nan,nan,nan,左灯罩(CXW-128-Q2),空,吸油烟机
4,1101000100520,2018-07-10,空,1-FAA-A303,Z1,空,空,20.0,国内,nan,nan,nan,nan,右灯罩(CXW-128-Q2),空,吸油烟机
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70387,112500040005A,2024-07-05,空,SAEG-N0011,F1,空,空,20.0,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
70388,112500040005B,2024-07-16,空,SAEG-N0011,F1,空,空,20.0,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
70389,112500040010A,2024-06-27,空,SAEF-A0010,F1,空,空,10.0,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP1,空,服务产品
70390,112500040018A,2024-06-25,空,SAEH-A0010,F1,空,空,10.0,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP3,空,服务产品


In [138]:
for index,row in df_2024_result.iterrows():
    df_2024_result.loc[index,'2025年11月'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年11月_自制'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='10.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年11月_外购'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='20.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年11月_外协'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='30.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年1-11月新建数量'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年1-11月新建数量_自制'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='10.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年1-11月新建数量_外购'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='20.0')]['物料编码'].nunique()
    df_2024_result.loc[index,'2025年1-11月新建数量_外协'] = df_2025_live[(df_2025_live['产品组描述'].isin(productgroup_map[row['产品集合']]))&(df_2025_live['国内/海外']=='国内')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='30.0')]['物料编码'].nunique()   
df_2024_result

,产品集合,2024年基准,2024年基准_自制,2024年基准_外购,2024年基准_外协,2025年11月,2025年11月_自制,2025年11月_外购,2025年11月_外协,2025年1-11月新建数量,2025年1-11月新建数量_自制,2025年1-11月新建数量_外购,2025年1-11月新建数量_外协
0,油烟机,11823.0,4188.0,7184.0,448.0,14302.0,5039.0,8719.0,544.0,3032.0,1020.0,1897.0,115.0
1,灶具,6485.0,1224.0,5196.0,65.0,7215.0,1377.0,5736.0,102.0,1026.0,228.0,755.0,43.0
2,蒸烤微合计,6655.0,1445.0,5006.0,203.0,7431.0,1643.0,5559.0,229.0,1094.0,318.0,738.0,38.0
3,灶集成,2060.0,603.0,1375.0,82.0,2516.0,659.0,1752.0,105.0,877.0,226.0,604.0,47.0
4,烹饪合计,15200.0,3272.0,11577.0,350.0,17162.0,3679.0,13047.0,436.0,2997.0,772.0,2097.0,128.0
5,水槽洗碗机,4088.0,520.0,3464.0,96.0,4472.0,594.0,3732.0,146.0,730.0,135.0,531.0,64.0
6,嵌入式洗碗机,2936.0,376.0,2398.0,162.0,3886.0,515.0,3144.0,227.0,1280.0,180.0,979.0,121.0
7,洗碗机合计,7024.0,896.0,5862.0,258.0,8358.0,1109.0,6876.0,373.0,2010.0,315.0,1510.0,185.0
8,家用净水机,2083.0,137.0,1868.0,78.0,2898.0,159.0,2595.0,144.0,974.0,47.0,854.0,73.0
9,热水器,4352.0,606.0,3644.0,102.0,4596.0,663.0,3807.0,126.0,488.0,106.0,338.0,44.0


In [139]:
temp_2024 = {'产品集合':'海外合计',
'2024年基准':df_2024_live[(df_2024_live['国内/海外']=='海外')&(df_2024_live['创建日期']<='2024-12-31')]['物料编码'].nunique(),
'2024年基准_自制':df_2024_live[(df_2024_live['国内/海外']=='海外')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='10.0')]['物料编码'].nunique(),
'2024年基准_外购':df_2024_live[(df_2024_live['国内/海外']=='海外')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='20.0')]['物料编码'].nunique(),
'2024年基准_外协':df_2024_live[(df_2024_live['国内/海外']=='海外')&(df_2024_live['创建日期']<='2024-12-31')&(df_2024_live['制造方式']=='30.0')]['物料编码'].nunique(),
'2025年11月':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')]['物料编码'].nunique(),
'2025年11月_自制':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='10.0')]['物料编码'].nunique(),
'2025年11月_外购':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='20.0')]['物料编码'].nunique(),
'2025年11月_外协':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['制造方式']=='30.0')]['物料编码'].nunique(),
'2025年1-11月新建数量':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')]['物料编码'].nunique(),
'2025年1-11月新建数量_自制':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='10.0')]['物料编码'].nunique(),
'2025年1-11月新建数量_外购':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='20.0')]['物料编码'].nunique(),
'2025年1-11月新建数量_外协':df_2025_live[(df_2025_live['国内/海外']=='海外')&(df_2025_live['创建日期']<='2025-11-31')&(df_2025_live['创建日期']>='2025-01-01')&(df_2025_live['制造方式']=='30.0')]['物料编码'].nunique(),
}
temp_2024_df = pd.DataFrame([temp_2024])
df_2024_result = pd.concat([df_2024_result,temp_2024_df],axis=0,ignore_index=True)
df_2024_result

,产品集合,2024年基准,2024年基准_自制,2024年基准_外购,2024年基准_外协,2025年11月,2025年11月_自制,2025年11月_外购,2025年11月_外协,2025年1-11月新建数量,2025年1-11月新建数量_自制,2025年1-11月新建数量_外购,2025年1-11月新建数量_外协
0,油烟机,11823.0,4188.0,7184.0,448.0,14302.0,5039.0,8719.0,544.0,3032.0,1020.0,1897.0,115.0
1,灶具,6485.0,1224.0,5196.0,65.0,7215.0,1377.0,5736.0,102.0,1026.0,228.0,755.0,43.0
2,蒸烤微合计,6655.0,1445.0,5006.0,203.0,7431.0,1643.0,5559.0,229.0,1094.0,318.0,738.0,38.0
3,灶集成,2060.0,603.0,1375.0,82.0,2516.0,659.0,1752.0,105.0,877.0,226.0,604.0,47.0
4,烹饪合计,15200.0,3272.0,11577.0,350.0,17162.0,3679.0,13047.0,436.0,2997.0,772.0,2097.0,128.0
5,水槽洗碗机,4088.0,520.0,3464.0,96.0,4472.0,594.0,3732.0,146.0,730.0,135.0,531.0,64.0
6,嵌入式洗碗机,2936.0,376.0,2398.0,162.0,3886.0,515.0,3144.0,227.0,1280.0,180.0,979.0,121.0
7,洗碗机合计,7024.0,896.0,5862.0,258.0,8358.0,1109.0,6876.0,373.0,2010.0,315.0,1510.0,185.0
8,家用净水机,2083.0,137.0,1868.0,78.0,2898.0,159.0,2595.0,144.0,974.0,47.0,854.0,73.0
9,热水器,4352.0,606.0,3644.0,102.0,4596.0,663.0,3807.0,126.0,488.0,106.0,338.0,44.0


In [140]:
df_2024_result.to_excel(r'C:\Users\zhangbon\Desktop\零件详情截至2025-11-30.xlsx', index=False)

### 读取原始数据，并开始处理

In [36]:
df = pd.read_excel(fr"D:\000物料报表\{month}\单物料产值\物料精简原始数据11月.XLSX")
df = df.astype(str)
#展示df的数据格式
df.info()
# 展示df的前五行数据
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125304 entries, 0 to 125303
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   物料编码         125304 non-null  object
 1   创建日期         125304 non-null  object
 2   是否冻结         125304 non-null  object
 3   图号           125304 non-null  object
 4   产品组          125304 non-null  object
 5   跨工厂物料状态      125304 non-null  object
 6   是否虚拟物料       125304 non-null  object
 7   国内/海外        125304 non-null  object
 8   是否涉及认证一致性管控  125304 non-null  object
 9   整机认证地区       125304 non-null  object
 10  整机认证类型       125304 non-null  object
 11  整机海外认证标准     125304 non-null  object
 12  物料描述         125304 non-null  object
dtypes: object(13)
memory usage: 12.4+ MB


,物料编码,创建日期,是否冻结,图号,产品组,跨工厂物料状态,是否虚拟物料,国内/海外,是否涉及认证一致性管控,整机认证地区,整机认证类型,整机海外认证标准,物料描述
0,1101000100400,2018-07-10,X,1-FAA-A004,Z1,nan,nan,20.0,nan,nan,nan,nan,垫脚(CXW-128-Q2)
1,1101000100410,2018-07-10,nan,1-FAA-A106,Z1,nan,nan,20.0,nan,nan,nan,nan,线夹(CXW-128-Q2)
2,1101000100430,2018-07-10,X,1-FAA-A200,Z1,nan,nan,20.0,nan,nan,nan,nan,上导风板组件(CXW-128-Q2)
3,1101000100440,2018-07-10,X,1-FAA-A201,Z1,nan,nan,20.0,nan,nan,nan,nan,连接圈(CXW-128-Q2)
4,1101000100450,2018-07-10,X,1-FAA-A211,Z1,nan,nan,20.0,nan,nan,nan,nan,上导风板(CXW-128-Q2)


In [37]:
list_drop = []
df['删除原因'] = '空'
for i in range(len(df)):
    if 'ZD70' in df.loc[i,'物料描述']:
        list_drop.append(i)
        df.loc[i,'删除原因'] = 'ODM'
    if df.loc[i,'图号'][:4] == 'LAFS':
        list_drop.append(i)
        df.loc[i,'删除原因'] = '火炬'
    if df.loc[i,'图号'][:4] == 'LAFT':
        list_drop.append(i)
        df.loc[i,'删除原因'] = '火炬'
    if df.loc[i,'物料编码'][:8] == '11010022':
        list_drop.append(i)
        df.loc[i,'删除原因'] = 'IMES'  
    if df.loc[i,'物料编码'][:8] == '11010027':
        list_drop.append(i)
        df.loc[i,'删除原因'] = '空调油烟机'  
    if df.loc[i,'图号'][:4] == 'PACA':
        list_drop.append(i)
        df.loc[i,'删除原因'] = '净存机'
    if df.loc[i,'产品组'] == 'ZN':
        list_drop.append(i)
        df.loc[i,'删除原因'] = '灶消'

In [38]:
#依据产品组的产品组描述的对照关系，生成一列产品组描述
# 输入：字典, 产品组描述的对照关系，产品组描述的列名
df['产品组描述'] = df['产品组'].map(product_dict)
df['国内/海外'] = df['国内/海外'].apply(lambda x:'国内' if x=='20.0' else ('海外' if x=='10.0' else '空'))
df = df.fillna('空')
df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']] = df[['是否冻结','跨工厂物料状态','是否虚拟物料','删除原因']].replace('nan','空')
df

,物料编码,创建日期,是否冻结,图号,产品组,跨工厂物料状态,是否虚拟物料,国内/海外,是否涉及认证一致性管控,整机认证地区,整机认证类型,整机海外认证标准,物料描述,删除原因,产品组描述
0,1101000100400,2018-07-10,X,1-FAA-A004,Z1,空,空,国内,nan,nan,nan,nan,垫脚(CXW-128-Q2),空,吸油烟机
1,1101000100410,2018-07-10,空,1-FAA-A106,Z1,空,空,国内,nan,nan,nan,nan,线夹(CXW-128-Q2),空,吸油烟机
2,1101000100430,2018-07-10,X,1-FAA-A200,Z1,空,空,国内,nan,nan,nan,nan,上导风板组件(CXW-128-Q2),空,吸油烟机
3,1101000100440,2018-07-10,X,1-FAA-A201,Z1,空,空,国内,nan,nan,nan,nan,连接圈(CXW-128-Q2),空,吸油烟机
4,1101000100450,2018-07-10,X,1-FAA-A211,Z1,空,空,国内,nan,nan,nan,nan,上导风板(CXW-128-Q2),空,吸油烟机
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125299,112500040005A,2024-07-05,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125300,112500040005B,2024-07-16,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125301,112500040010A,2024-06-27,空,SAEF-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP1,空,服务产品
125302,112500040018A,2024-06-25,空,SAEH-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP3,空,服务产品


In [39]:
df.to_excel(fr"D:\000物料报表\{month}\单物料产值\物料精简原始数据处理稿.xlsx",index=False)

In [40]:
#先筛出删除原因为空的df,且非虚拟物料
df_not_del = df[(df["删除原因"] == "空")&(df["是否虚拟物料"] == "空")]
# df_not_del

### 对未冻结的物料做计算处理

In [41]:
#先对未冻结的做处理
df_not_froze = df_not_del[df_not_del["是否冻结"] == "空"]
df_not_froze["创建日期"] = pd.to_datetime(df_not_froze["创建日期"])
df_not_froze


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\878873356.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_not_froze["创建日期"] = pd.to_datetime(df_not_froze["创建日期"])


,物料编码,创建日期,是否冻结,图号,产品组,跨工厂物料状态,是否虚拟物料,国内/海外,是否涉及认证一致性管控,整机认证地区,整机认证类型,整机海外认证标准,物料描述,删除原因,产品组描述
1,1101000100410,2018-07-10,空,1-FAA-A106,Z1,空,空,国内,nan,nan,nan,nan,线夹(CXW-128-Q2),空,吸油烟机
5,1101000100470,2018-07-10,空,1-FAA-A221,Z1,空,空,国内,nan,nan,nan,nan,网圈(CXW-128-Q2),空,吸油烟机
6,1101000100480,2018-07-10,空,1-FAA-A222,Z1,空,空,国内,nan,nan,nan,nan,网罩内圈(CXW-128-Q2),空,吸油烟机
8,1101000100510,2018-07-10,空,1-FAA-A302,Z1,空,空,国内,nan,nan,nan,nan,左灯罩(CXW-128-Q2),空,吸油烟机
9,1101000100520,2018-07-10,空,1-FAA-A303,Z1,空,空,国内,nan,nan,nan,nan,右灯罩(CXW-128-Q2),空,吸油烟机
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125299,112500040005A,2024-07-05,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125300,112500040005B,2024-07-16,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125301,112500040010A,2024-06-27,空,SAEF-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP1,空,服务产品
125302,112500040018A,2024-06-25,空,SAEH-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP3,空,服务产品


In [42]:
#对未冻结的国内的物料进行处理,all代表当前周期内的所有物料数，part代表的是周期内所有的新增物料数
df_not_froze_china = df_not_froze[df_not_froze["国内/海外"] == "国内"]
#对于国内的所有进行处理
df_not_froze_china_all = df_not_froze_china[df_not_froze_china['创建日期'] <= end_date ].groupby(['产品组描述']).size().reset_index(name='数量')
df_not_froze_china_part = df_not_froze_china[(df_not_froze_china['创建日期'] >= start_date ) & (df_not_froze_china['创建日期'] <= end_date )].groupby(['产品组描述']).size().reset_index(name='数量')
with pd.ExcelWriter(fr"D:\000物料报表\{month}\单物料产值\非冻结计数—国内.xlsx") as writer:
    df_not_froze_china_all.to_excel(writer, sheet_name='国内-all', index=False)
    df_not_froze_china_part.to_excel(writer, sheet_name='国内-part', index=False)
df_not_froze_china_all

,产品组描述,数量
0,下排集成灶,1560
1,两用炉,454
2,保温箱,84
3,吸油烟机,14059
4,商用净水机,734
5,地面手持式清洁机,1839
6,外置循环系统,72
7,大厨管家,1
8,家用冰箱,1628
9,家用净水机,2854


In [43]:
#对未冻结的国外的物料进行处理
df_not_froze_oversea = df_not_froze[df_not_froze["国内/海外"] == "海外"]
df_not_froze_oversea_all = df_not_froze_oversea[df_not_froze_oversea['创建日期'] <= end_date ].groupby(['产品组描述']).size().reset_index(name='数量')
df_not_froze_oversea_part = df_not_froze_oversea[(df_not_froze_oversea['创建日期'] >= start_date ) & (df_not_froze_oversea['创建日期'] <= end_date )].groupby(['产品组描述']).size().reset_index(name='数量')
with pd.ExcelWriter(fr"D:\000物料报表\{month}\单物料产值\非冻结计数—海外.xlsx") as writer:
    df_not_froze_oversea_all.to_excel(writer, sheet_name='海外-all', index=False)
    df_not_froze_oversea_part.to_excel(writer, sheet_name='海外-part', index=False)
print(f'海外物料总数量为：{sum(df_not_froze_oversea_all["数量"])}')
print(f'海外物料限定范围内数量为：{sum(df_not_froze_oversea_part["数量"])}')
print(f'海外物料总数量(去除不考核产品)为：{sum(df_not_froze_oversea_all[~(df_not_froze_oversea_all["产品组描述"]=="不考核产品")]["数量"])}')
print(f'海外物料限定范围内数量(去除不考核产品)为：{sum(df_not_froze_oversea_part[~(df_not_froze_oversea_part["产品组描述"]=="不考核产品")]["数量"])}')


海外物料总数量为：8834
海外物料限定范围内数量为：1299
海外物料总数量(去除不考核产品)为：8834
海外物料限定范围内数量(去除不考核产品)为：1299


### 对剔除掉冻结或采购冻结的物料进行处理

In [44]:
#筛选出跨工厂物料状态为空且是否冻结为空
df_not_caigou_froze = df_not_del[(df_not_del['跨工厂物料状态'] == '空') & (df_not_del['是否冻结'] == "空")]
df_not_caigou_froze["创建日期"] = pd.to_datetime(df_not_caigou_froze["创建日期"])
df_not_caigou_froze


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\1626582603.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_not_caigou_froze["创建日期"] = pd.to_datetime(df_not_caigou_froze["创建日期"])


,物料编码,创建日期,是否冻结,图号,产品组,跨工厂物料状态,是否虚拟物料,国内/海外,是否涉及认证一致性管控,整机认证地区,整机认证类型,整机海外认证标准,物料描述,删除原因,产品组描述
1,1101000100410,2018-07-10,空,1-FAA-A106,Z1,空,空,国内,nan,nan,nan,nan,线夹(CXW-128-Q2),空,吸油烟机
5,1101000100470,2018-07-10,空,1-FAA-A221,Z1,空,空,国内,nan,nan,nan,nan,网圈(CXW-128-Q2),空,吸油烟机
6,1101000100480,2018-07-10,空,1-FAA-A222,Z1,空,空,国内,nan,nan,nan,nan,网罩内圈(CXW-128-Q2),空,吸油烟机
8,1101000100510,2018-07-10,空,1-FAA-A302,Z1,空,空,国内,nan,nan,nan,nan,左灯罩(CXW-128-Q2),空,吸油烟机
9,1101000100520,2018-07-10,空,1-FAA-A303,Z1,空,空,国内,nan,nan,nan,nan,右灯罩(CXW-128-Q2),空,吸油烟机
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125299,112500040005A,2024-07-05,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125300,112500040005B,2024-07-16,空,SAEG-N0011,F1,空,空,国内,nan,nan,nan,nan,附件包装盒_542*382*44_FJ-YKZP2,空,服务产品
125301,112500040010A,2024-06-27,空,SAEF-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP1,空,服务产品
125302,112500040018A,2024-06-25,空,SAEH-A0010,F1,空,空,国内,nan,nan,nan,nan,蒸烤盘_银色_FJ-YKZP3,空,服务产品


In [45]:
#对非采购冻结和冻结的国内物料进行处理
df_not_caigou_froze_china = df_not_caigou_froze[df_not_caigou_froze["国内/海外"] == "国内"]
df_not_caigou_froze_china['创建日期'] = pd.to_datetime(df_not_caigou_froze_china['创建日期'])
#对于国内的进行处理
df_not_caigou_froze_china_all = df_not_caigou_froze_china[df_not_caigou_froze_china['创建日期'] <= end_date ].groupby(['产品组描述']).size().reset_index(name='数量')
df_not_caigou_froze_china_part = df_not_caigou_froze_china[(df_not_caigou_froze_china['创建日期'] >= start_date ) & (df_not_caigou_froze_china['创建日期'] <= end_date )].groupby(['产品组描述']).size().reset_index(name='数量')
with pd.ExcelWriter(fr"D:\000物料报表\{month}\单物料产值\非采购非冻结计数—国内.xlsx") as writer:
    df_not_caigou_froze_china_all.to_excel(writer, sheet_name='国内-all', index=False)
    df_not_caigou_froze_china_part.to_excel(writer, sheet_name='国内-part', index=False)
df_not_caigou_froze_china_all

#建立产品组描述和数量之间的映射关系
productgroup_all_china_map = dict(zip(df_not_caigou_froze_china_all['产品组描述'], df_not_caigou_froze_china_all['数量']))
productgroup_part_china_map = dict(zip(df_not_caigou_froze_china_part['产品组描述'], df_not_caigou_froze_china_part['数量']))
productgroup_all_china_map 


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\527062125.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_not_caigou_froze_china['创建日期'] = pd.to_datetime(df_not_caigou_froze_china['创建日期'])


{'下排集成灶': 1560,
 '两用炉': 451,
 '保温箱': 84,
 '吸油烟机': 12578,
 '商用净水机': 734,
 '地面手持式清洁机': 1839,
 '外置循环系统': 72,
 '大厨管家': 1,
 '家用冰箱': 1628,
 '家用净水机': 2854,
 '嵌入式洗碗机': 3768,
 '微波炉': 564,
 '料理机': 274,
 '新风空净': 581,
 '服务产品': 201,
 '气电烹饪灶（智能灶）': 158,
 '水槽洗碗机': 4407,
 '消毒柜': 2789,
 '灶具': 6554,
 '灶蒸烤烹饪机': 2113,
 '灶蒸烹饪机': 91,
 '烤箱': 1227,
 '热水器': 3965,
 '蒸微': 423,
 '蒸烤微烹饪机': 732,
 '蒸烤烹饪机': 2568,
 '蒸箱': 1502,
 '蒸锅': 36,
 '集成式洗碗机': 1047}

In [46]:
#对于海外的进行处理
df_not_caigou_froze_oversea = df_not_caigou_froze[df_not_caigou_froze["国内/海外"] == "海外"]
df_not_caigou_froze_oversea["创建日期"] = pd.to_datetime(df_not_caigou_froze_oversea['创建日期'])
#对于海外的所有进行处理
df_not_caigou_froze_oversea_all = df_not_caigou_froze_oversea[df_not_caigou_froze_oversea['创建日期'] <= end_date ].groupby(['产品组描述']).size().reset_index(name='数量')
#对于海外的限定时间期限进行处理
df_not_caigou_froze_oversea_part = df_not_caigou_froze_oversea[(df_not_caigou_froze_oversea['创建日期'] >= start_date ) & (df_not_caigou_froze_oversea['创建日期'] <= end_date )].groupby(['产品组描述']).size().reset_index(name='数量')
with pd.ExcelWriter(fr"D:\000物料报表\{month}\单物料产值\非采购非冻结计数—海外.xlsx") as writer:
    df_not_caigou_froze_oversea_all.to_excel(writer, sheet_name='海外-all', index=False)
    df_not_caigou_froze_oversea_part.to_excel(writer, sheet_name='海外-part', index=False)
df_not_caigou_froze_oversea_all
print(f'非采购非冻结海外物料总数量为（未排除不统计的产品组数据）：{sum(df_not_caigou_froze_oversea_all["数量"])}')
print(f'非采购非冻结海外物料限定范围内数量为（未排除不统计的产品组数据）：{sum(df_not_caigou_froze_oversea_part["数量"])}')
#建立产品组描述和数量之间的映射关系
productgroup_all_oversea_map = sum(df_not_caigou_froze_oversea_all[(df_not_caigou_froze_oversea_all["产品组描述"].isin(productgroup_map['国内产品线合计']))]["数量"])
productgroup_part_oversea_map = sum(df_not_caigou_froze_oversea_part[(df_not_caigou_froze_oversea_part["产品组描述"].isin(productgroup_map['国内产品线合计']))]["数量"])
print(productgroup_all_oversea_map)
print(productgroup_part_oversea_map)


非采购非冻结海外物料总数量为（未排除不统计的产品组数据）：8833
非采购非冻结海外物料限定范围内数量为（未排除不统计的产品组数据）：1299
8765
1282


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\2579071116.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_not_caigou_froze_oversea["创建日期"] = pd.to_datetime(df_not_caigou_froze_oversea['创建日期'])


### 构造出产品集合，每个产品集合对应的现状物料数量和统计周期内的新增物料数量

In [47]:

# 匹配出每个统计维度的所有个数
productgrouped2_count_all = {}
for key,value in productgroup_map.items():
    for v in value:
        productgrouped2_count_all[key] = productgrouped2_count_all.get(key,0) + productgroup_all_china_map.get(v,0)
productgrouped2_count_all['海外合计'] = productgroup_all_oversea_map
print(productgrouped2_count_all)

#匹配出统计周期内的新增个数
productgrouped2_count_part = {}
for key,value in productgroup_map.items():
    for v in value:
        productgrouped2_count_part[key] = productgrouped2_count_part.get(key,0) + productgroup_part_china_map.get(v,0)
productgrouped2_count_part['海外合计'] = productgroup_part_oversea_map
print(productgrouped2_count_part)


{'油烟机': 12578, '灶具': 6554, '蒸烤微合计': 7016, '灶集成': 2204, '烹饪合计': 15774, '水槽洗碗机': 4407, '嵌入式洗碗机': 3768, '洗碗机合计': 8175, '家用净水机': 2854, '热水器': 4416, '净热合计': 7270, '消毒柜': 2789, '国内产品线合计': 46586, '海外合计': 8765}
{'油烟机': 2787, '灶具': 887, '蒸烤微合计': 948, '灶集成': 787, '烹饪合计': 2622, '水槽洗碗机': 665, '嵌入式洗碗机': 1162, '洗碗机合计': 1827, '家用净水机': 930, '热水器': 366, '净热合计': 1296, '消毒柜': 261, '国内产品线合计': 8793, '海外合计': 1282}


### 构建基准数据表

In [48]:

#构建目标数据的DataFrame首列
df_out = pd.DataFrame()
df_out['产品线'] = ['油烟机产品线', '烹饪厨电产品线', '烹饪厨电产品线', '烹饪厨电产品线', '烹饪厨电产品线', '洗碗机产品线', '洗碗机产品线', '洗碗机产品线', '净热产品线', '净热产品线', '净热产品线', '冰储产品线', '国内产品线合计', '海外合计']
df_out['产品集合']=['油烟机', '灶具', '蒸烤微合计', '灶集成', '烹饪合计', '水槽洗碗机', '嵌入式洗碗机', '洗碗机合计', '家用净水机', '热水器', '净热合计', '消毒柜', '国内产品线合计', '海外合计']
df_out


,产品线,产品集合
0,油烟机产品线,油烟机
1,烹饪厨电产品线,灶具
2,烹饪厨电产品线,蒸烤微合计
3,烹饪厨电产品线,灶集成
4,烹饪厨电产品线,烹饪合计
5,洗碗机产品线,水槽洗碗机
6,洗碗机产品线,嵌入式洗碗机
7,洗碗机产品线,洗碗机合计
8,净热产品线,家用净水机
9,净热产品线,热水器


In [49]:
df_out[f'{year_name}基准零部件数'] = df_out['产品集合'].map(productgroup_starting_map)
df_out[f'{year_name+month_name}零部件数'] = df_out['产品集合'].map(productgrouped2_count_all)
df_out[f'{year_name}1-{month_name}新建数量'] = df_out['产品集合'].map(productgrouped2_count_part)
df_out['净增加数量'] = df_out[f'{year_name+month_name}零部件数'] - df_out[f'{year_name}基准零部件数']
df_out[f'{year_name}冻结数量'] = df_out[f'{year_name}1-{month_name}新建数量'] - df_out['净增加数量']
df_out['物料精简率'] = (-df_out['净增加数量'] / df_out[f'{year_name}基准零部件数'] * 100).round(2).apply(lambda x: f"{x:.2f}%")
df_out[f'{income_year_name}'] = df_out['产品集合'].map(productgroup_sales_map)
df_out[f'{year_name+month_name}发货收入'] =  df_out['产品集合'].map(productgroup2_income_map)
df_out['基准单物料产值'] = (df_out[f'{income_year_name}'] / df_out[f'{year_name}基准零部件数']).round(2)
df_out[f'{year_name+month_name}单物料产值'] = (df_out[f'{year_name+month_name}发货收入'] / df_out[f'{year_name+month_name}零部件数']).round(2)
df_out['目标'] = df_out['产品集合'].map(target_map)
df_out['发货收入增加率'] =  ((df_out[f'{year_name+month_name}发货收入'] - df_out[f'{income_year_name}']) / df_out[f'{income_year_name}'] * 100).round(2).apply(lambda x: f"{x:.2f}%")
df_out['产值变化率'] = ((df_out[f'{year_name+month_name}单物料产值'] - df_out['基准单物料产值']) / df_out['基准单物料产值'] * 100).round(2).apply(lambda x: f"{x:.2f}%")
df_out['仍需冻结物料数'] = '/'
df_out['同目标差异'] = '/'
df_out = df_out.fillna('/')
for index,row in df_out.iterrows():
    if row['目标'] == '/':
        df_out.loc[index,'仍需冻结物料数'] = '/'
        df_out.loc[index,'同目标差异'] = '/'
    else:
        df_out.loc[index,'仍需冻结物料数'] = df_out.loc[index,f'{year_name+month_name}零部件数'] - df_out.loc[index,f'{year_name+month_name}发货收入'] / df_out.loc[index,'目标']
        df_out.loc[index,'同目标差异'] = df_out.loc[index,f'{year_name+month_name}单物料产值'] - float(df_out.loc[index,'目标'])
        if df_out.loc[index,'仍需冻结物料数'] < 0:
            df_out.loc[index,'仍需冻结物料数'] = '已达标'
        else:
            df_out.loc[index,'仍需冻结物料数'] = np.ceil(df_out.loc[index, '仍需冻结物料数']).astype(int)
df_out['备注'] = df_out['产品集合'].map(beizhu)

df_out



,产品线,产品集合,2025年基准零部件数,2025年11月零部件数,2025年1-11月新建数量,净增加数量,2025年冻结数量,物料精简率,2024年发货收入,2025年11月发货收入,基准单物料产值,2025年11月单物料产值,目标,发货收入增加率,产值变化率,仍需冻结物料数,同目标差异,备注
0,油烟机产品线,油烟机,11823,12578,2787,755,2032,-6.39%,760275.0,737320.8298,64.3,58.62,58.5,-3.02%,-8.83%,已达标,0.12,不含ODM、空调/IMES
1,烹饪厨电产品线,灶具,6485,6554,887,69,818,-1.06%,364114.0,366186.4071,56.15,55.87,49.52,0.57%,-0.50%,已达标,6.35,不含火炬、智能灶
2,烹饪厨电产品线,蒸烤微合计,6655,7016,948,361,587,-5.42%,117582.0,114481.049,17.67,16.32,16.29,-2.64%,-7.64%,已达标,0.03,含烹饪机
3,烹饪厨电产品线,灶集成,2060,2204,787,144,643,-6.99%,51586.0,64578.592,25.04,29.3,24.89,25.19%,17.01%,已达标,4.41,不含灶消
4,烹饪厨电产品线,烹饪合计,15200,15774,2622,574,2048,-3.78%,533281.0,545246.0481,35.08,34.57,31.49,2.24%,-1.45%,已达标,3.08,不含灶消
5,洗碗机产品线,水槽洗碗机,4088,4407,665,319,346,-7.80%,70840.0,76989.7925,17.33,17.47,/,8.68%,0.81%,/,/,/
6,洗碗机产品线,嵌入式洗碗机,2936,3768,1162,832,330,-28.34%,129987.0,142664.318,44.27,37.86,/,9.75%,-14.48%,/,/,/
7,洗碗机产品线,洗碗机合计,7024,8175,1827,1151,676,-16.39%,200827.0,219654.1105,28.59,26.87,25.73,9.37%,-6.02%,已达标,1.14,/
8,净热产品线,家用净水机,2083,2854,930,771,159,-37.01%,21298.0,21954.001,10.22,7.69,9.2,3.08%,-24.76%,468,-1.51,/
9,净热产品线,热水器,4352,4416,366,64,302,-1.47%,50544.0,48246.6023,11.61,10.93,10.45,-4.55%,-5.86%,已达标,0.48,含两用炉


### 产品线单物料产值（表格）

In [50]:
df_out1 = df_out[df_out['产品集合'].isin(['油烟机','烹饪合计','洗碗机合计','净热合计','消毒柜'])]
df_out1['同目标差异'] = df_out1[f'{year_name+month_name}单物料产值'] - df_out1['目标']
df_out1 = df_out1[['产品线','产品集合','基准单物料产值',f'{year_name+month_name}单物料产值','目标','同目标差异','产值变化率']]
display(df_out1)


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_2904\151329981.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_out1['同目标差异'] = df_out1[f'{year_name+month_name}单物料产值'] - df_out1['目标']


,产品线,产品集合,基准单物料产值,2025年11月单物料产值,目标,同目标差异,产值变化率
0,油烟机产品线,油烟机,64.3,58.62,58.5,0.12,-8.83%
4,烹饪厨电产品线,烹饪合计,35.08,34.57,31.49,3.08,-1.45%
7,洗碗机产品线,洗碗机合计,28.59,26.87,25.73,1.14,-6.02%
10,净热产品线,净热合计,11.16,9.66,10.04,-0.38,-13.44%
11,冰储产品线,消毒柜,19.87,14.78,14.5,0.28,-25.62%


### 产品组单物料产值

In [51]:
df_out2 = df_out[['产品线',	'产品集合','基准单物料产值','目标',	f'{year_name}{month_name}单物料产值','同目标差异','产值变化率','物料精简率','备注',	'仍需冻结物料数']]
df_out2 = df_out2.rename(columns={'仍需冻结物料数': '是否达标'})
df_out2


,产品线,产品集合,基准单物料产值,目标,2025年11月单物料产值,同目标差异,产值变化率,物料精简率,备注,是否达标
0,油烟机产品线,油烟机,64.3,58.5,58.62,0.12,-8.83%,-6.39%,不含ODM、空调/IMES,已达标
1,烹饪厨电产品线,灶具,56.15,49.52,55.87,6.35,-0.50%,-1.06%,不含火炬、智能灶,已达标
2,烹饪厨电产品线,蒸烤微合计,17.67,16.29,16.32,0.03,-7.64%,-5.42%,含烹饪机,已达标
3,烹饪厨电产品线,灶集成,25.04,24.89,29.3,4.41,17.01%,-6.99%,不含灶消,已达标
4,烹饪厨电产品线,烹饪合计,35.08,31.49,34.57,3.08,-1.45%,-3.78%,不含灶消,已达标
5,洗碗机产品线,水槽洗碗机,17.33,/,17.47,/,0.81%,-7.80%,/,/
6,洗碗机产品线,嵌入式洗碗机,44.27,/,37.86,/,-14.48%,-28.34%,/,/
7,洗碗机产品线,洗碗机合计,28.59,25.73,26.87,1.14,-6.02%,-16.39%,/,已达标
8,净热产品线,家用净水机,10.22,9.2,7.69,-1.51,-24.76%,-37.01%,/,468
9,净热产品线,热水器,11.61,10.45,10.93,0.48,-5.86%,-1.47%,含两用炉,已达标


## 物料精简率

### 产品线精简率

In [52]:
df_out3 = df_out.loc[df_out['产品集合'].isin(['油烟机','烹饪合计','洗碗机合计','净热合计','消毒柜','国内产品线合计','海外合计']),['产品线','产品集合','物料精简率']]
df_out3

,产品线,产品集合,物料精简率
0,油烟机产品线,油烟机,-6.39%
4,烹饪厨电产品线,烹饪合计,-3.78%
7,洗碗机产品线,洗碗机合计,-16.39%
10,净热产品线,净热合计,-12.98%
11,冰储产品线,消毒柜,-7.72%
12,国内产品线合计,国内产品线合计,-8.16%
13,海外合计,海外合计,-16.14%


### 产品组精简率

In [53]:

df_out4 = df_out[['产品线', '产品集合', f'{year_name}基准零部件数' ,f'{year_name+month_name}零部件数',f'{year_name}1-{month_name}新建数量',f'{year_name}冻结数量','净增加数量','物料精简率','备注']]
df_out4

,产品线,产品集合,2025年基准零部件数,2025年11月零部件数,2025年1-11月新建数量,2025年冻结数量,净增加数量,物料精简率,备注
0,油烟机产品线,油烟机,11823,12578,2787,2032,755,-6.39%,不含ODM、空调/IMES
1,烹饪厨电产品线,灶具,6485,6554,887,818,69,-1.06%,不含火炬、智能灶
2,烹饪厨电产品线,蒸烤微合计,6655,7016,948,587,361,-5.42%,含烹饪机
3,烹饪厨电产品线,灶集成,2060,2204,787,643,144,-6.99%,不含灶消
4,烹饪厨电产品线,烹饪合计,15200,15774,2622,2048,574,-3.78%,不含灶消
5,洗碗机产品线,水槽洗碗机,4088,4407,665,346,319,-7.80%,/
6,洗碗机产品线,嵌入式洗碗机,2936,3768,1162,330,832,-28.34%,/
7,洗碗机产品线,洗碗机合计,7024,8175,1827,676,1151,-16.39%,/
8,净热产品线,家用净水机,2083,2854,930,159,771,-37.01%,/
9,净热产品线,热水器,4352,4416,366,302,64,-1.47%,含两用炉


# 结果导出

In [54]:
#将df_out,df_out1,df_out2放到3个sheet并导出
with pd.ExcelWriter(rf'D:\000物料报表\{month}\单物料产值\物料精简报表.xlsx') as writer:
    df_out.to_excel(writer, sheet_name='sheet1', index=False)
    df_out1.to_excel(writer, sheet_name='产品线单物料产值', index=False)
    df_out2.to_excel(writer, sheet_name='产品组单物料产值', index=False)
    df_out3.to_excel(writer, sheet_name='产品线精简率', index=False)
    df_out4.to_excel(writer, sheet_name='产品组精简率', index=False)


